In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

In [2]:
df = pd.read_csv(
    "opi_builder_c_live_record.csv"
)

print(df.shape)

print()

print(df["Label"].value_counts())

(7013, 12)

Label
NORMAL    6436
ATTACK     577
Name: count, dtype: int64


In [3]:
df["Label"] = df["Label"].map({

    "NORMAL":0,
    "ATTACK":1

})

print(
    df["Label"].value_counts()
)

Label
0    6436
1     577
Name: count, dtype: int64


In [4]:
FEATURES = [

    "Flow Duration",

    "Flow Bytes/s",

    "Flow Packets/s",

    "Total Fwd Packets",

    "Total Length of Fwd Packets",

    "Fwd Packets/s",

    "SYN Flag Count",

    "ACK Flag Count",

    "Init_Win_bytes_forward",

    "act_data_pkt_fwd"

]

X = df[FEATURES]

y = df["Label"]

print(X.shape)
print(y.shape)

(7013, 10)
(7013,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [6]:
print("TRAIN")

print(
    y_train.value_counts()
)

print()

print("TEST")

print(
    y_test.value_counts()
)

TRAIN
Label
0    5148
1     462
Name: count, dtype: int64

TEST
Label
0    1288
1     115
Name: count, dtype: int64


In [7]:
rf = RandomForestClassifier(

    n_estimators=200,

    random_state=42,

    class_weight="balanced",

    n_jobs=-1

)

rf.fit(
    X_train,
    y_train
)

print("Training Finished")

Training Finished


In [8]:
y_pred = rf.predict(X_test)

In [9]:
acc = accuracy_score(
    y_test,
    y_pred
)

print()

print(
    f"Accuracy = {acc:.4f}"
)

print()

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

print()

print(

    classification_report(

        y_test,
        y_pred,

        target_names=[
            "NORMAL",
            "ATTACK"
        ]

    )

)


Accuracy = 1.0000

[[1288    0]
 [   0  115]]

              precision    recall  f1-score   support

      NORMAL       1.00      1.00      1.00      1288
      ATTACK       1.00      1.00      1.00       115

    accuracy                           1.00      1403
   macro avg       1.00      1.00      1.00      1403
weighted avg       1.00      1.00      1.00      1403



In [10]:
importance = pd.DataFrame({

    "Feature":FEATURES,

    "Importance":rf.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

importance

,Feature,Importance
7,ACK Flag Count,0.314041
6,SYN Flag Count,0.239310
9,act_data_pkt_fwd,0.143190
8,Init_Win_bytes_forward,0.084968
1,Flow Bytes/s,0.081173
4,Total Length of Fwd Packets,0.047049
3,Total Fwd Packets,0.044179
0,Flow Duration,0.033573
5,Fwd Packets/s,0.006993
2,Flow Packets/s,0.005524


In [11]:
joblib.dump(

    rf,

    "rf_opi_live.joblib"

)

print()

print(
    "Saved : rf_opi_live.joblib"
)


Saved : rf_opi_live.joblib


In [12]:
import os

size_mb = (

    os.path.getsize(
        "rf_opi_live.joblib"
    )

    / 1024 / 1024

)

print(
    f"Model Size = {size_mb:.2f} MB"
)

Model Size = 0.37 MB


In [13]:
print(df.groupby(["Source_File","Label"]).size())

Source_File           Label
normal_browsing.csv   0        1454
normal_download.csv   0        1536
normal_gaming.csv     0         698
normal_mixed.csv      0        1284
normal_streaming.csv  0        1464
syn_flood_basic.csv   1         116
syn_flood_randip.csv  1         130
syn_randip_basic.csv  1         110
syn_u100_basic.csv    1         104
syn_u10_basic.csv     1         117
dtype: int64
